# 1. What you'll learn

        - read a GP posterior mean and uncertainty band
- see how kernel length scale encodes smoothness
- run expected-improvement Bayesian optimization against random search

        **The one question this notebook answers:** How can a model say both what it predicts and where it knows too little, then use that uncertainty to choose the next experiment?

# 2. Setup

A one-dimensional objective keeps every probability visible. Scikit-learn supplies exact Gaussian-process regression; SciPy supplies the normal CDF and density used by expected improvement.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The hidden objective is a smooth wavy bowl. We begin with five noisy evaluations. x is the controllable setting; y is the costly measured loss we want to minimize.

**Small example:** Suppose five losses are [4.0, 1.8, 0.9, 1.2, 3.5]. The current best is 0.9. A candidate predicting 0.8±0.05 is safe; one predicting 1.0±0.8 may still have larger expected improvement because uncertainty leaves room below 0.9.

In [ ]:
def objective(x): return .18*(x-1.6)**2+np.sin(2.7*x)+.15*np.cos(7*x)
rng=np.random.default_rng(SEED);grid=np.linspace(-2.5,4.5,500)[:,None]
x_obs=np.array([-2.2,-.7,.8,2.5,4.2])[:,None];y_obs=objective(x_obs[:,0])+rng.normal(0,.08,len(x_obs))
true_y=objective(grid[:,0]);true_x=grid[np.argmin(true_y),0]
print("Observed shape:",x_obs.shape,"current best:",round(y_obs.min(),3),"teaching-only true minimum x:",round(true_x,3))

# 4. Explore

We inspect the true landscape for teaching, the sparse initial observations available to the model, and kernel correlations at three length scales.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.6));axes[0].plot(grid,true_y);axes[0].set_title("Hidden objective")
axes[1].plot(grid,true_y,color=".8");axes[1].scatter(x_obs,y_obs,color="#176b66",s=45);axes[1].set_title("Only five costly observations")
d=np.linspace(0,3,150);[axes[2].plot(d,np.exp(-d**2/(2*l*l)),label=f"length={l}") for l in [.2,.7,1.5]];axes[2].set(title="RBF-style correlation",xlabel="distance",ylabel="similarity");axes[2].legend();plt.tight_layout();plt.show()

**Takeaway:** The objective has several local dips, so blindly descending from one point can miss the best basin.

**Takeaway:** Five observations leave large gaps where honest uncertainty should widen.

**Takeaway:** A short length scale permits rapid wiggles; a long one couples distant x values and produces smoother beliefs.

# 5. Prepare

There is no conventional random split in the optimization loop: each queried point becomes training data. For honest regression evaluation, the dense true grid is used only after fitting and never to choose model parameters.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
kernel=ConstantKernel(1.0,(.1,10))*Matern(length_scale=.8,length_scale_bounds=(.1,3),nu=2.5)+WhiteKernel(.01,(1e-5,.2))
def fit_gp(x,y): return GaussianProcessRegressor(kernel=kernel,normalize_y=True,n_restarts_optimizer=1,random_state=SEED).fit(x,y)
gp=fit_gp(x_obs,y_obs);print("Initial kernel learned:",gp.kernel_)

# 6. Train

        Conditioning a GP turns prior correlations into a posterior mean and variance. Bayesian optimization refits that posterior after every measurement and queries where expected improvement balances promise with ignorance.

        Conceptually, fitting follows these steps:

        1. Build a covariance matrix from distances and the kernel.
2. Condition the joint Gaussian on observed y values.
3. Predict mean μ and standard deviation σ over candidates.
4. Maximize expected improvement, observe that x, append it, and repeat.

In [ ]:
def expected_improvement(mu,sigma,best):
    gain=best-mu-.01;z=np.divide(gain,sigma,out=np.zeros_like(gain),where=sigma>1e-12)
    return gain*norm.cdf(z)+sigma*norm.pdf(z)
x_bo=x_obs.copy();y_bo=y_obs.copy();history=[]
for step in range(8):
    model=fit_gp(x_bo,y_bo);mu,std=model.predict(grid,return_std=True);ei=expected_improvement(mu,std,y_bo.min());idx=int(np.argmax(ei))
    x_new=grid[idx:idx+1];y_new=objective(x_new[:,0])+rng.normal(0,.08,1);x_bo=np.vstack([x_bo,x_new]);y_bo=np.r_[y_bo,y_new];history.append((float(x_new[0,0]),float(y_new[0])))
print("Queries:",[(round(x,2),round(y,2)) for x,y in history]);print("Best observed:",round(y_bo.min(),3))

# 7. Evaluate

We compare posterior RMSE with a constant-mean baseline and compare the best value after each Bayesian query with 50 seeded random-search runs using the same budget.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
final_gp=fit_gp(x_bo,y_bo);mu,std=final_gp.predict(grid,return_std=True);baseline=np.repeat(y_obs.mean(),len(grid));print(f"GP RMSE={mean_squared_error(true_y,mu)**.5:.3f}; mean baseline={mean_squared_error(true_y,baseline)**.5:.3f}")
bo_curve=np.minimum.accumulate(y_bo)[5:];random_curves=[]
for s in range(50):
    rr=np.random.default_rng(s);xs=rr.uniform(-2.5,4.5,8);ys=objective(xs);random_curves.append(np.minimum.accumulate(np.r_[y_obs.min(),ys])[1:])
random_mean=np.mean(random_curves,axis=0);fig,axes=plt.subplots(1,3,figsize=(14,3.7))
axes[0].plot(grid,mu,label="posterior mean");axes[0].fill_between(grid[:,0],mu-1.96*std,mu+1.96*std,alpha=.25);axes[0].scatter(x_bo,y_bo,s=28,color="#176b66");axes[0].plot(grid,true_y,"--",color="black",alpha=.55);axes[0].set_title("Posterior and 95% band")
axes[1].plot(range(1,9),bo_curve,marker="o",label="Bayesian optimization");axes[1].plot(range(1,9),random_mean,marker="o",label="random mean");axes[1].axhline(true_y.min(),ls="--",color="black");axes[1].set(title="Best loss so far",xlabel="new evaluations");axes[1].legend()
axes[2].plot(grid,final_gp.predict(grid,return_std=True)[1],color="#d38b45");axes[2].scatter(x_bo,np.zeros(len(x_bo)),marker="|",color="black");axes[2].set(title="Uncertainty falls near observations",xlabel="x",ylabel="posterior σ")
plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The GP uncertainty contracts around observations, and expected improvement usually locates a near-global minimum in fewer evaluations than uniform random search—when its smoothness story is credible.

# 8. Break it

We force an excessively long length scale and almost no noise. The posterior becomes smoothly overconfident, erasing real wiggles and producing confidence intervals that miss the true objective.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
bad_kernel=ConstantKernel(1.0)*Matern(length_scale=8,length_scale_bounds="fixed",nu=2.5)+WhiteKernel(1e-6,noise_level_bounds="fixed")
bad=GaussianProcessRegressor(kernel=bad_kernel,normalize_y=True,random_state=SEED).fit(x_obs,y_obs);bad_mu,bad_std=bad.predict(grid,return_std=True)
coverage=np.mean((true_y>=bad_mu-1.96*bad_std)&(true_y<=bad_mu+1.96*bad_std));good_coverage=np.mean((true_y>=mu-1.96*std)&(true_y<=mu+1.96*std))
print(f"Over-smooth 95% coverage={coverage:.3f}; tuned GP coverage={good_coverage:.3f}; bad RMSE={mean_squared_error(true_y,bad_mu)**.5:.3f}")

**Use this when…** evaluations are expensive, data are small, and calibrated uncertainty can guide the next measurement.

        **Don't use this when…** you have huge data, abrupt nonstationary structure, many irrelevant dimensions, or a cheap objective that random/grid search can evaluate freely.

        ## Try this

        1. Start with three clustered points; expect uncertainty and exploration to increase elsewhere.
2. Replace Matern ν=2.5 with RBF; compare smoothness and coverage.
3. Raise the observation noise to 0.4 and see expected improvement explore longer.